# Chapitre 0.3 : installer son poste

Avant d’entraîner une IA, il faut savoir sur quoi elle va calculer. Dans ce carnet, tu regardes la machine qui le fait tourner comme la verront les projets de La Forge IA : le pilote de la carte graphique, la version de PyTorch, la carte qu’il voit, et les fils du processeur. Puis tu fais un vrai calcul, sur la carte si elle est là.

Ce carnet accompagne [la page du chapitre 0.3](https://chipat-neko.github.io/forge-ia/partie-0/0-3/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. demander au pilote NVIDIA quelle carte il voit, avec `nvidia-smi` ;
2. relever ce que voient Python et PyTorch ;
3. lire le diagnostic, et ce qu’il conseille de faire ;
4. faire un essai de calcul, sur la carte si elle est prête ;
5. relever un défi, puis vérifier que ce carnet donne le même diagnostic que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer : Colab a déjà Python et PyTorch, et ce carnet ne les installe pas. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ».

**Pour avoir une carte graphique dans Colab** : menu « Exécution », puis « Modifier le type d’exécution ». Choisis un accélérateur « GPU », puis enregistre. Colab redémarre : relance les cellules depuis le début. L’offre gratuite ne garantit pas de carte, et une session dure au plus 12 heures. Sans carte, tout marche aussi, sur le processeur.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet utilise PyTorch, que Colab a déjà installé.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "0.3-poste"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))
print("Le code du projet est prêt.")

## Étape 1 : la carte, vue par son pilote

`nvidia-smi` est un programme livré avec le pilote NVIDIA. En haut, il affiche la version du pilote, et la version de CUDA la plus haute que ce pilote accepte (« CUDA Version », ou « CUDA UMD Version » avec les pilotes les plus récents). En dessous : chaque carte, sa mémoire, et les programmes qui s’en servent.

Sans pilote NVIDIA, `nvidia-smi` n’existe pas : la cellule le dit, au lieu de s’arrêter sur une erreur. Le bloc `try` / `except` attrape cette erreur pour l’expliquer.

In [ ]:
import subprocess

try:
    resultat = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
except FileNotFoundError:
    print("nvidia-smi est introuvable\u202f: pas de pilote NVIDIA ici, donc pas de carte NVIDIA à utiliser.")
    print("Dans Colab, pour avoir une carte\u202f: Exécution > Modifier le type d’exécution > GPU.")
except subprocess.TimeoutExpired:
    print("nvidia-smi ne répond pas\u202f: le pilote NVIDIA semble bloqué. Redémarre la machine, puis relance.")
else:
    if resultat.returncode == 0:
        print(resultat.stdout)
    else:
        print(f"nvidia-smi répond par une erreur\u202f: {(resultat.stderr or resultat.stdout).strip()}")

## Étape 2 : ce que voient Python et PyTorch

`relever_faits()` regarde le poste sans rien changer :

- la version de PyTorch : « +cu130 » à la fin veut dire « faite pour CUDA 13.0 », et « +cpu », « pour le processeur seul » ;
- `torch.version.cuda`, la version de CUDA livrée avec PyTorch (`None` pour la version processeur) ;
- `torch.cuda.is_available()`, qui dit si PyTorch voit une carte ;
- le nom de la carte et sa **capacité de calcul**, sa génération : 7.5 pour une T4, 12.0 pour une RTX 5070 Ti ;
- les architectures pour lesquelles ce PyTorch a été compilé, écrites « sm_75 » pour la capacité 7.5 ; chacune sert aussi les cartes de sa génération, de mineure plus grande ;
- le pilote, lu dans `nvidia-smi` ;
- les fils de calcul du processeur, et ceux que PyTorch utilise.

In [ ]:
from poste.faits import decrire_faits, relever_faits

faits = relever_faits()
for ligne in decrire_faits(faits):
    print(ligne)

## Étape 3 : le diagnostic

`diagnostiquer(faits)` lit ces faits, sans rien lancer, et en tire des constats. Chacun a un niveau : « bon », « attention » (ça marche, mais moins bien que ça pourrait) ou « problème » (il faut agir), et dit ce qu’il faut faire.

Les commandes `uv` qu’il conseille se lancent sur ton ordinateur, dans le dossier du projet. Dans Colab, il n’y a rien à installer : c’est le constat qui compte.

In [ ]:
from poste.diagnostic import Niveau, decrire_constat, diagnostiquer

constats = diagnostiquer(faits)
for constat in constats:
    for ligne in decrire_constat(constat):
        print(ligne)

## Étape 4 : un essai de calcul

Un vrai calcul, pour finir : deux matrices de 512 × 512 nombres, multipliées en float32, les nombres à virgule de 32 bits que les IA utilisent. Le même produit, fait en float64, deux fois plus précis, sert de référence : l’écart doit être minuscule. La cellule mesure aussi la durée d’un produit, en millisecondes.

L’essai se fait sur le processeur, puis sur la carte si le diagnostic la dit prête.

In [ ]:
from poste.essai import decrire_essai, essai_de_calcul

print(decrire_essai(essai_de_calcul("cpu")))
carte_prete = any(c.cle == "carte_prete" for c in constats)
if carte_prete:
    print(decrire_essai(essai_de_calcul("cuda")))
else:
    print("Pas d’essai sur une carte graphique\u202f: PyTorch n’en voit aucune de prête ici.")
    print("Dans Colab, pour avoir une carte\u202f: Exécution > Modifier le type d’exécution > GPU, puis relance le carnet depuis le début.")

## Exercice : le nom d’une architecture

PyTorch nomme ses architectures d’après la capacité des cartes : « sm_75 » pour la capacité 7.5 (une T4), « sm_120 » pour la capacité 12.0 (une RTX 5070 Ti).

Une carte est connue de PyTorch si sa génération est dans la liste, avec une mineure plus petite ou égale : aucune version de PyTorch n’a « sm_89 », mais « sm_86 » sert une RTX 4070, de capacité 8.9. C’est ce que regarde le diagnostic, avec la fonction `carte_connue`, pour savoir si une carte est trop récente ou trop ancienne pour sa version de PyTorch.

Écris la fonction `nom_d_architecture(capacite)`. Elle reçoit une capacité, un couple de deux entiers comme `(7, 5)`, et renvoie son nom, comme `"sm_75"`. La cellule l’essaie sur la T4 et sur la RTX 5070 Ti. Si elle trouve les deux, le défi est réussi. Elle regarde alors, pour trois cartes du chapitre et pour une RTX 4070, si le nom de la carte est dans la liste de sa version de PyTorch, et si `carte_connue` la dit connue.

Indice : `majeure, mineure = capacite` sépare les deux nombres.

In [ ]:
from poste.diagnostic import ARCHITECTURES_CU132, SITUATIONS, carte_connue


def nom_d_architecture(capacite: tuple[int, int]) -> str:
    # À toi : (7, 5) donne "sm_75", et (12, 0) donne "sm_120".
    return ""


# Ne change rien en dessous : cette partie essaie ta fonction sur la T4 et sur la RTX 5070 Ti.
t4, rtx = nom_d_architecture((7, 5)), nom_d_architecture((12, 0))
if (t4, rtx) == ("sm_75", "sm_120"):
    print("Défi réussi\u202f: (7, 5) donne sm_75, et (12, 0) donne sm_120.")
    cartes = [SITUATIONS[nom] for nom in ["windows_cu132_rtx5070ti", "windows_cu126_rtx5070ti", "colab_t4"]]
    cartes = [(f.carte, f.capacite, f.torch, f.architectures) for f in cartes]
    cartes.append(("NVIDIA GeForce RTX 4070", (8, 9), "2.15.0+cu132", ARCHITECTURES_CU132))
    for carte, capacite, version, architectures in cartes:
        nom = nom_d_architecture(capacite)
        dans = "est" if nom in architectures else "n’est pas"
        connue = "connue" if carte_connue(capacite, architectures) else "inconnue"
        print(f"{carte} et PyTorch {version}\u202f: {nom} {dans} dans la liste, et la carte est {connue}.")
else:
    print(f"Pas encore. Ta fonction donne {t4!r} pour (7, 5) et {rtx!r} pour (12, 0), au lieu de «\u202fsm_75\u202f» et «\u202fsm_120\u202f».")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

La capacité (12, 0) a pour majeure 12 et pour mineure 0 : on les écrit l’une après l’autre, derrière « sm_ ».

Remplace la fonction de la cellule de l’exercice par :

```python
def nom_d_architecture(capacite: tuple[int, int]) -> str:
    majeure, mineure = capacite
    return f"sm_{majeure}{mineure}"
```

Le nom seul ne suffit pas toujours : une RTX 4070 (sm_89) est connue de PyTorch 2.15 grâce à sm_86, de la même génération 8. C’est pourquoi `carte_connue`, dans le projet, compare la génération et la mineure, pas seulement le nom.

</details>

## Dernière étape : le même diagnostic que la page ?

La page du site ne lance pas Python : son tableau des situations lit un fichier de référence, `tests/attendus.json`, écrit par le diagnostic du projet. La cellule suivante refait le diagnostic de chaque situation et le compare au fichier, caractère par caractère. Puis elle dit sur quoi ce carnet a tourné, et compte les problèmes de ce poste.

In [ ]:
import json

from poste.attendus import calculer_attendus, vers_json

fichier = (projet / "tests" / "attendus.json").read_text(encoding="utf-8")
print(f"{len(json.loads(fichier)['situations'])} situations comparées.")
print(f"Même diagnostic que la page du site\u202f: {'oui' if vers_json(calculer_attendus()) == fichier else 'non'}")

if carte_prete:
    print(f"Ce carnet a tourné sur\u202f: la carte graphique, {faits.carte}.")
else:
    print("Ce carnet a tourné sur\u202f: le processeur.")
problemes = sum(c.niveau == Niveau.PROBLEME for c in constats)
print(f"Diagnostic terminé\u202f: {problemes} problème(s).")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/0.3-poste sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/0.3-poste).
- La page du chapitre, pour installer ton poste pas à pas, faire les paris et lire le code expliqué : [chapitre 0.3](https://chipat-neko.github.io/forge-ia/partie-0/0-3/).
- Le défi du chapitre : lance aussi `python -m poste` sur ton ordinateur, dans le dossier `projets/0.3-poste` (`uv run --extra gpu python -m poste` avec une carte NVIDIA, `uv run --extra cpu python -m poste` sans), puis compare son rapport à celui de ce carnet.
- Le chapitre 0.4 manipule les tenseurs de PyTorch, sur le processeur et sur la carte.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.